Використай цей шаблон в роботі з датасетом.      
Ти можеш додавати комірки за потреби, але не змінюй структуру і послідовність питань.      
Обмежся функціями з наведених бібліотек.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

1. Завантаж датасет з бібліотеки seaborn:

In [ ]:
# Завантажуємо датасет Titanic із seaborn
# Примітка: seaborn вперше завантажує прикладні датасети з інтернету та кешує локально.
# Якщо інтернету немає, спробуємо підхопити файл з локального кешу seaborn.
from seaborn.utils import get_data_home
import os

try:
    df = sns.load_dataset('titanic')
except Exception as e:
    cache_path = os.path.join(get_data_home(), 'titanic.csv')
    if os.path.exists(cache_path):
        df = pd.read_csv(cache_path)
        print("Датасет завантажено з локального кешу seaborn:", cache_path)
    else:
        raise RuntimeError(
            "Не вдалося завантажити датасет через sns.load_dataset('titanic'). "
            "Перевір підключення до інтернету (потрібно для першого завантаження) "
            "або наявність кешу seaborn у каталозі: " + get_data_home()
        ) from e

df


2. Переглянь перші рядки датасету. Зроби висновок, чи коректно він завантажився.

In [ ]:
# Перші 5 рядків датасету
display(df.head())
print(f"Shape: {df.shape}")

3. Перевір типи стовпців. Які з них потребують перетворення?

In [ ]:
# Типи даних у стовпцях
dtypes = df.dtypes.to_frame('dtype')
display(dtypes)

print("\nКоментар:")
print("- Частина числових ознак (наприклад, pclass, survived) за змістом є категоріальними, але їх можна залишати int для розрахунків.")
print("- 'deck' та 'class' уже мають тип category; 'adult_male' та 'alone' — bool; інші текстові — object.")

4. Перевір статистику по УСІМ стовпцям датасету.

In [ ]:
# Статистика по всіх стовпцях (включно з категоріальними)
stats_all = df.describe(include='all').T
display(stats_all)

5. Необхідно створити єдиний стовпчик, що вказує кількість родичів для кожного пасажира на борту, замість:
- кількість братів/сестер або чоловіків/дружин на борту;
- кількість батьків або дітей на борту;

Булева ознака: стовпчик 'alone'=True, якщо пасажир подорожував один (без родичів на борту).        
Перевір, щоб для одиноких пасажирів новий стовпчик мав значення 0.      
Після створення нового стовпчика, дропни "sibsp", "parch", "alone".

In [ ]:
# 5) Створюємо єдиний стовпчик кількості родичів на борту
# relatives = sibsp + parch
df['relatives'] = df['sibsp'] + df['parch']

# Перевірка: якщо пасажир "alone" == True, то relatives має бути 0
# (Якщо є розбіжності — подивимось, але не очікуємо їх у цьому датасеті)
mismatch = df.loc[df['alone'] & (df['relatives'] != 0), ['sibsp', 'parch', 'alone', 'relatives']]
display(mismatch.head())
print("К-сть можливих розбіжностей:", len(mismatch))

# Дропаємо зайві колонки
df = df.drop(columns=['sibsp', 'parch', 'alone'])
display(df.head())

6. Перевір, як часто зустрічається певна кількість родичів в новому стовпці.    
Результат представ у вигляді таблиці, побудованої з використанням групування та агрегації:

In [ ]:
# 6) Як часто зустрічається певна кількість родичів
relatives_freq = (
    df.groupby('relatives')
      .agg(passengers=('survived', 'size'),
           survived=('survived', 'sum'),
           survived_rate=('survived', 'mean'))
      .reset_index()
      .sort_values('relatives')
)

# Відсоток виживання зробимо більш читабельним
relatives_freq['survived_rate'] = (relatives_freq['survived_rate'] * 100).round(1)

display(relatives_freq)

7. Використовуючи цикл заміни кількість родичів, що перевищує число 5(п'ять) на значення "above 5"(понад п'ять).       
Запиши значення в новий стовпчик ʼrelatives_categoryʼ.

In [ ]:
# 7) Категоризація кількості родичів (понад 5 -> 'above 5') у новий стовпець relatives_category
relatives_category = []

for r in df['relatives']:
    if r > 5:
        relatives_category.append('above 5')
    else:
        relatives_category.append(str(int(r)))

df['relatives_category'] = relatives_category
display(df[['relatives', 'relatives_category']].head(10))

8. Необхідно вивести на екран статистику по відсотковому показнику пасажирів з кількістю родичів більше 5 відносно загального числа пасажирів для кожного з міст посадки. Для цього:
- порахуй загальну кількість пасажирів в кожному з міст посадки,
- знайди число пасажирів з кількістю родичів більше 5 в кожному з міст посадки,
- поділи ці два стовчики між собою, перетворивши результат у відсотки (ціле число).        
Відобрази статистику для усіх міст посадки.

In [ ]:
# 8) Відсоток пасажирів з кількістю родичів > 5 відносно загальної кількості по містах посадки
# Використаємо embark_town (назва міста). Для NaN зробимо категорію 'Unknown'
df['embark_town_filled'] = df['embark_town'].fillna('Unknown')

total_by_town = df.groupby('embark_town_filled').size().rename('total_passengers')
above5_by_town = df[df['relatives'] > 5].groupby('embark_town_filled').size().rename('above5_passengers')

town_stats = pd.concat([total_by_town, above5_by_town], axis=1).fillna(0)
town_stats['above5_passengers'] = town_stats['above5_passengers'].astype(int)
town_stats['above5_percent'] = ((town_stats['above5_passengers'] / town_stats['total_passengers']) * 100).round().astype('Int64')

display(town_stats.reset_index().rename(columns={'embark_town_filled': 'embark_town'}))

9. Заповни відсутні значення віку медіаною.

In [ ]:
# 9) Заповнюємо пропуски у віці медіаною
median_age = df['age'].median()
print("Median age:", median_age)

missing_before = df['age'].isna().sum()
df['age'] = df['age'].fillna(median_age)
missing_after = df['age'].isna().sum()

print(f"Missing age before: {missing_before}")
print(f"Missing age after:  {missing_after}")

10. Створи новий стовпець, де вік представлено категорією, замість числа (наприклад: до 14 років, 14-34 роки, 35-59 років, 60 і більше років). Виконай задачу з використанням користувацької функції. Осіб з невідомим віком познач відповідно.

In [ ]:
# 10) Створюємо вікову категорію за допомогою користувацької функції
def age_to_category(age):
    if pd.isna(age):
        return 'unknown'
    if age < 14:
        return '0-13'
    elif age < 35:
        return '14-34'
    elif age < 60:
        return '35-59'
    else:
        return '60+'

df['age_category'] = df['age'].apply(age_to_category)

display(df[['age', 'age_category']].head(10))
print(df['age_category'].value_counts())

11. Перевір, в якій віковій категорії була найвища смертність.     
Для цього рекомендується перетворити стовпець 'alive' в булевий тип.    
 Потім підрахувати загальну кількість пасажирів та кількість тих, хто не вижив.      
 Потім обчисли відносний показниках для кожної категорії.

In [ ]:
# 11) У якій віковій категорії була найвища смертність?
# alive -> bool: True якщо вижив
df['alive_bool'] = df['alive'].map({'yes': True, 'no': False})

mortality_by_age_cat = (
    df.groupby('age_category')
      .agg(total=('alive_bool', 'size'),
           dead=('alive_bool', lambda s: (~s).sum()))
)

mortality_by_age_cat['mortality_rate_%'] = (mortality_by_age_cat['dead'] / mortality_by_age_cat['total'] * 100).round(1)
mortality_by_age_cat = mortality_by_age_cat.sort_values('mortality_rate_%', ascending=False)

display(mortality_by_age_cat)

print("\nНайвища смертність у категорії:", mortality_by_age_cat.index[0])

*Бонусне завдання*         :          
Підготуй розгорнуту статистику смертності по категорії віку, класу квитка, рівню каюти та кількості родичів.      
Які фактори, на твою думку, найбільше пов'язані з рівнем смерності? (наприклад: найбільша смертність у відсотковому значенні спостерігається серед вікової групи ... класу квитка.... при наявності ... родичів та для рівня каюти.... Фактор ... має найвищий вплив на смертність)

In [ ]:
# Бонус: розгорнута статистика смертності за віком, класом, палубою та кількістю родичів

# Підготуємо допоміжні колонки
df['deck_filled'] = df['deck'].astype('object').fillna('Unknown')  # deck - category з NaN
df['class_filled'] = df['class'].astype('object')                  # зручно для групувань

# 1) Загальна смертність по ключових зрізах
overall_mortality = (1 - df['alive_bool'].mean()) * 100
print(f"Загальна смертність у датасеті: {overall_mortality:.1f}%")

# 2) Смертність за віковою категорією
mort_age = mortality_by_age_cat.copy()
display(mort_age)

# 3) Смертність за класом квитка
mort_class = (
    df.groupby('class_filled')
      .agg(total=('alive_bool', 'size'),
           dead=('alive_bool', lambda s: (~s).sum()))
)
mort_class['mortality_rate_%'] = (mort_class['dead'] / mort_class['total'] * 100).round(1)
mort_class = mort_class.sort_values('mortality_rate_%', ascending=False)
display(mort_class)

# 4) Смертність за рівнем каюти (палубою)
mort_deck = (
    df.groupby('deck_filled')
      .agg(total=('alive_bool', 'size'),
           dead=('alive_bool', lambda s: (~s).sum()))
)
mort_deck['mortality_rate_%'] = (mort_deck['dead'] / mort_deck['total'] * 100).round(1)
mort_deck = mort_deck.sort_values('mortality_rate_%', ascending=False)
display(mort_deck)

# 5) Смертність за кількістю родичів (категорія)
mort_rel_cat = (
    df.groupby('relatives_category')
      .agg(total=('alive_bool', 'size'),
           dead=('alive_bool', lambda s: (~s).sum()))
)
mort_rel_cat['mortality_rate_%'] = (mort_rel_cat['dead'] / mort_rel_cat['total'] * 100).round(1)
mort_rel_cat = mort_rel_cat.sort_values('mortality_rate_%', ascending=False)
display(mort_rel_cat)

# 6) Перехресні таблиці (для більш глибокого аналізу)
# Вікова категорія x клас
pivot_age_class = pd.pivot_table(
    df, index='age_category', columns='class_filled',
    values='alive_bool', aggfunc=lambda s: (1 - s.mean()) * 100
).round(1)
display(pivot_age_class)

# Палуба x клас (обережно: мало спостережень у деяких палубах)
pivot_deck_class = pd.pivot_table(
    df, index='deck_filled', columns='class_filled',
    values='alive_bool', aggfunc=lambda s: (1 - s.mean()) * 100
).round(1)
display(pivot_deck_class)

# 7) Візуалізації
plt.figure(figsize=(8, 4))
sns.barplot(data=mort_class.reset_index(), x='class_filled', y='mortality_rate_%')
plt.title('Смертність за класом квитка (%)')
plt.xlabel('Клас')
plt.ylabel('Смертність, %')
plt.show()

plt.figure(figsize=(8, 4))
sns.barplot(data=mort_rel_cat.reset_index(), x='relatives_category', y='mortality_rate_%')
plt.title('Смертність за кількістю родичів (%)')
plt.xlabel('Категорія кількості родичів')
plt.ylabel('Смертність, %')
plt.show()

plt.figure(figsize=(9, 5))
sns.heatmap(pivot_age_class, annot=True, fmt=".1f")
plt.title('Смертність (%) за віком та класом квитка')
plt.xlabel('Клас')
plt.ylabel('Вікова категорія')
plt.show()

# 8) Висновок (коротко)
top_age = mort_age.index[0]
top_class = mort_class.index[0]
top_rel = mort_rel_cat.index[0]
top_deck = mort_deck.index[0]

print("\nВисновок:")
print(f"- Найвища смертність за віком: {top_age}")
print(f"- Найвища смертність за класом квитка: {top_class}")
print(f"- Найвища смертність за кількістю родичів: {top_rel}")
print(f"- Найвища смертність за палубою: {top_deck}")
print("\nЗа цим датасетом найсильніше з смертністю пов'язані соціально-економічний статус (клас квитка) "
      "та (опосередковано) доступність місця/палуби; також помітна різниця між віковими групами. "
      "Ефект кількості родичів більш неоднорідний і залежить від того, чи пасажир подорожував один/з сім'єю.")